In [0]:
# Databricks Notebook: 03_Gold_Aggregate
# Gold 层：silver → gold 业务聚合，面向分析/报表
# 幂等策略：CREATE OR REPLACE（重跑结果一致）
# 性能约定：新表使用 Liquid Clustering（DBR 15.4+，禁用分区/Z-ORDER）

from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
print(f"📅 Gold 聚合日期: {today}")

# =====================================================
# 1. gold.daily_move_summary：每日生产活跃度
#    面试考点：报废 = qty_out < qty_in（track out 时少了片）
# =====================================================
spark.sql("""
CREATE OR REPLACE TABLE gold.daily_move_summary
CLUSTER BY (move_date) AS
SELECT move_date,
       COUNT(DISTINCT lot_id)                                    AS active_lots,
       COUNT(*)                                                  AS move_count,
       SUM(CASE WHEN qty_out < qty_in THEN qty_in - qty_out ELSE 0 END) AS scrap_qty
FROM silver.moves
GROUP BY move_date
""")
print("✅ gold.daily_move_summary（每日活跃 lot / 过站数 / 报废片数）")

# =====================================================
# 2. gold.daily_cp_yield：每日 CP 良率（按产品）
#    良率 = BIN1(PASS) die 占比 —— CP 发生在 route 结束后
# =====================================================
spark.sql("""
CREATE OR REPLACE TABLE gold.daily_cp_yield
CLUSTER BY (test_date) AS
SELECT test_date,
       product,
       SUM(die_count)                                            AS total_dies,
       SUM(CASE WHEN bin_code = 'BIN1' THEN die_count ELSE 0 END) AS pass_dies,
       ROUND(SUM(CASE WHEN bin_code = 'BIN1' THEN die_count ELSE 0 END) * 100.0
             / SUM(die_count), 2)                                AS cp_yield_pct
FROM silver.cp_bins
GROUP BY test_date, product
""")
print("✅ gold.daily_cp_yield（每日每产品 CP 良率）")

# =====================================================
# 3. gold.equip_oee_daily：设备 OEE
#    可用率 = RUN 时间占比；结合过站数与报废率做简化 OEE
#    OEE = Availability × Performance × Quality
# =====================================================
spark.sql("""
CREATE OR REPLACE TABLE gold.equip_oee_daily
CLUSTER BY (state_date) AS
SELECT s.state_date,
       s.equipment_id,
       ROUND(100.0 * SUM(CASE WHEN s.state = 'RUN' THEN s.duration_min ELSE 0 END)
             / SUM(s.duration_min), 2)                           AS availability_pct,
       COALESCE(m.move_count, 0)                                 AS move_count,
       COALESCE(m.scrap_rate_pct, 0.0)                           AS scrap_rate_pct,
       ROUND(100.0 * SUM(CASE WHEN s.state = 'DOWN' THEN s.duration_min ELSE 0 END)
             / SUM(s.duration_min), 2)                           AS downtime_pct
FROM silver.equip_state s
LEFT JOIN (
    SELECT move_date, equipment_id,
           COUNT(*) AS move_count,
           ROUND(SUM(CASE WHEN qty_out < qty_in THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
               AS scrap_rate_pct
    FROM silver.moves
    GROUP BY move_date, equipment_id
) m
  ON s.equipment_id = m.equipment_id AND s.state_date = m.move_date
GROUP BY s.state_date, s.equipment_id, m.move_count, m.scrap_rate_pct
""")
print("✅ gold.equip_oee_daily（设备可用率/过站/停机）")

# =====================================================
# 4. gold.defect_pareto：缺陷帕累托（质量管理）
# =====================================================
spark.sql("""
CREATE OR REPLACE TABLE gold.defect_pareto
CLUSTER BY (inspection_date) AS
SELECT inspection_date,
       primary_defect,
       COUNT(*)                 AS lot_count,
       SUM(defect_count)        AS total_defects
FROM silver.quality
WHERE primary_defect IS NOT NULL
GROUP BY inspection_date, primary_defect
""")
print("✅ gold.defect_pareto（每日缺陷类型分布）")

# =====================================================
# 5. 验证：预览当天结果
# =====================================================
print("=" * 50)
print("当天 Gold 结果预览")
print("=" * 50)
spark.sql(f"""
    SELECT * FROM gold.daily_cp_yield WHERE test_date = '{today}'
""").show(truncate=False)
spark.sql(f"""
    SELECT * FROM gold.equip_oee_daily WHERE state_date = '{today}' LIMIT 5
""").show(truncate=False)

print("🎉 Gold 聚合完成！CREATE OR REPLACE 保证重跑幂等")


📅 Gold 聚合日期: 2026-09-03
✅ gold.daily_move_summary（每日活跃 lot / 过站数 / 报废片数）
✅ gold.daily_cp_yield（每日每产品 CP 良率）
✅ gold.equip_oee_daily（设备可用率/过站/停机）
✅ gold.defect_pareto（每日缺陷类型分布）
当天 Gold 结果预览
+---------+-------+----------+---------+------------+
|test_date|product|total_dies|pass_dies|cp_yield_pct|
+---------+-------+----------+---------+------------+
+---------+-------+----------+---------+------------+

+----------+------------+----------------+----------+--------------+------------+
|state_date|equipment_id|availability_pct|move_count|scrap_rate_pct|downtime_pct|
+----------+------------+----------------+----------+--------------+------------+
|2026-09-03|EQ-D01      |74.01           |0         |0.00          |14.73       |
|2026-09-03|EQ-C01      |67.55           |97        |1.03          |3.41        |
|2026-09-03|EQ-B01      |63.52           |63        |1.59          |0.00        |
|2026-09-03|EQ-F01      |37.18           |79        |1.27          |10.15       |
|2026-09-03|EQ-H02   